In [3]:
#this scripts uses calculated thermo cac equilibrium data to train ML models to predict D_max for different compositions. This uses CBFV as additional features.


In [13]:
#Import necessary libraries
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
from CBFV import composition
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import re
from ast import literal_eval
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from xgboost import XGBRegressor
from sklearn.multioutput import MultiOutputRegressor
from ax.service.ax_client import AxClient, ObjectiveProperties
from ax.core.arm import Arm
import ujson as js
from scipy.stats import sem

In [2]:
#according to the single tree notebook the best temperature ranges to utalize are: 2200, 1700, 2450, 2100, 2400, 1900, 1850
#according to the same notebook the NF or phase fraction features are ineffective at predicting D_max and thus will not be used

In [3]:
#pull training data
train_opt_CALPHAD_df = pd.read_csv(r"Data/F(Composition)_Data/CALPHAD_alloys_train_opt.csv")

#pull test data
test_CALPHAD_df = pd.read_csv(r"Data/F(Composition)_Data/CALPHAD_alloys_test.csv")


train_opt_CALPHAD_df.head()

,alloy_string,DF_AG2CA_T0C,DF_AG2CA_T1000C,DF_AG2CA_T100C,DF_AG2CA_T1050C,DF_AG2CA_T1100C,DF_AG2CA_T1150C,DF_AG2CA_T1200C,DF_AG2CA_T1250C,DF_AG2CA_T1300C,...,NF_ZRSI_T50C,NF_ZRSI_T550C,NF_ZRSI_T600C,NF_ZRSI_T650C,NF_ZRSI_T700C,NF_ZRSI_T750C,NF_ZRSI_T800C,NF_ZRSI_T850C,NF_ZRSI_T900C,NF_ZRSI_T950C
0,B22.00Co4.00Fe68.00Y6.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,Cu47.00Nb11.00Ni8.00Si1.00Ti33.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,B6.00C15.00Co6.42Er0.75Fe57.82Mo14.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Be24.00Cu12.00Fe8.00Nb8.00Zr48.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,Al10.00Ce60.00Cu20.00Ni10.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [4]:
#create the CBFV features for the training and test data
#start by pulling the formula column
train_opt_formula_df = pd.DataFrame({'formula': train_opt_CALPHAD_df['alloy_string']})
test_formula_df = pd.DataFrame({'formula': test_CALPHAD_df['alloy_string']})

#add a target column for CBFV api
train_opt_formula_df['target'] = 0
test_formula_df['target'] = 0

#use the CBFV api to create the features
train_opt_CBFV_df, _, train_opt_formulae, skipped_train = composition.generate_features(train_opt_formula_df, elem_prop='magpie')
test_CBFV_df, _, test_formulae, skipped_test = composition.generate_features(test_formula_df, elem_prop='magpie')


print(f"CBFV train features shape: {train_opt_CBFV_df.shape}, test features shape: {test_CBFV_df.shape}")
print(f"Number of skipped formulas: {len(skipped_train)}, {len(skipped_test)}")
train_opt_CBFV_df.head()


Processing Input Data: 100%|██████████| 882/882 [00:00<00:00, 27555.87it/s]


	Featurizing Compositions...


Assigning Features...: 100%|██████████| 882/882 [00:00<00:00, 27128.69it/s]


	Creating Pandas Objects...


Processing Input Data: 100%|██████████| 98/98 [00:00<00:00, 49026.93it/s]


	Featurizing Compositions...


Assigning Features...: 100%|██████████| 98/98 [00:00<00:00, 28005.85it/s]

	Creating Pandas Objects...
CBFV train features shape: (882, 132), test features shape: (98, 132)
Number of skipped formulas: 0, 0


,avg_Number,avg_MendeleevNumber,avg_AtomicWeight,avg_MeltingT,avg_Column,avg_Row,avg_CovalentRadius,avg_Electronegativity,avg_NsValence,avg_NpValence,...,mode_NValence,mode_NsUnfilled,mode_NpUnfilled,mode_NdUnfilled,mode_NfUnfilled,mode_NUnfilled,mode_GSvolume_pa,mode_GSbandgap,mode_GSmagmom,mode_SpaceGroupNumber
0,22.200000,56.280000,48.044699,1926.70000,8.840000,3.620000,124.680000,1.841600,2.000000,0.220000,...,8.0,0.0,0.0,4.0,0.0,4.0,10.730,0.0,2.110663,229.0
1,27.780000,55.100000,60.858759,1736.29190,7.980000,4.100000,143.910000,1.749000,1.420000,0.020000,...,11.0,1.0,0.0,0.0,0.0,1.0,11.070,0.0,0.000000,225.0
2,24.359036,58.662966,53.217519,2293.90019,8.946795,3.734973,123.841484,1.995602,1.859986,0.360036,...,8.0,0.0,0.0,4.0,0.0,4.0,10.730,0.0,2.110663,229.0
3,29.000000,53.040000,65.476074,1923.65240,4.760000,4.080000,146.560000,1.517600,1.800000,0.000000,...,4.0,0.0,0.0,8.0,0.0,8.0,23.195,0.0,0.000000,194.0
4,44.700000,35.200000,105.346294,1180.30100,6.300000,5.100000,173.300000,1.404000,1.800000,0.100000,...,4.0,0.0,0.0,9.0,13.0,22.0,37.240,0.0,0.000000,194.0


In [5]:
#filter the CALPHAD data to include only driving forces at the temepratures of interests
# Define the temperature ranges of interest
temperature_range = [2200, 1700, 2450, 2100, 2400, 1900, 1850]
temperature_range_str = [str(temp) for temp in temperature_range]

# Filter the columns to include only those with the specified temperature ranges and Df
filtered_CALPHAD_cols = [col for col in train_opt_CALPHAD_df.columns if any(temp in col for temp in temperature_range_str) and 'DF' in col]

print(f"Number of filtered CALPHAD columns: {len(filtered_CALPHAD_cols)}")

Number of filtered CALPHAD columns: 5628


In [ ]:
#combine the CBFV features with the filtered CALPHAD features for training and test data
X_train_opt = pd.concat([train_opt_CBFV_df, train_opt_CALPHAD_df[filtered_CALPHAD_cols]], axis=1)
X_test = pd.concat([test_CBFV_df, test_CALPHAD_df[filtered_CALPHAD_cols]], axis=1)

print(f"Combined train features shape: {X_train_opt.shape}, Combined test features shape: {X_test.shape}")


Combined train features shape: (882, 5760), Combined test features shape: (98, 5760)


In [9]:
#load the dmax json data and create y data
D_max_dict = js.load(open(r"Data\dmax_data.json", 'r'))

y_train_opt = [D_max_dict[formula] for formula in train_opt_CALPHAD_df['alloy_string']]
y_test = [D_max_dict[formula] for formula in test_CALPHAD_df['alloy_string']]



In [11]:
# Create 5 CV groups by clustering similar alloys together
# Standardize features for clustering
kmeans_scaler = StandardScaler()
kmeans_X_scaled = kmeans_scaler.fit_transform(X_train_opt)

# Use KMeans to group similar alloys into 5 clusters
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
cv_groups = kmeans.fit_predict(kmeans_X_scaled)

# Add group assignments to a dataframe for reference
cv_group_df = pd.DataFrame({
    'formula': train_opt_CALPHAD_df['alloy_string'],
    'cv_group': cv_groups
})

print("CV Group distribution:")
print(cv_group_df['cv_group'].value_counts().sort_index())
print(f"\nTotal samples: {len(cv_groups)}")
cv_group_df.head(10)

CV Group distribution:
cv_group
0    512
1    146
2     86
3     96
4     42
Name: count, dtype: int64

Total samples: 882


,formula,cv_group
0,B22.00Co4.00Fe68.00Y6.00,0
1,Cu47.00Nb11.00Ni8.00Si1.00Ti33.00,0
2,B6.00C15.00Co6.42Er0.75Fe57.82Mo14.00,2
3,Be24.00Cu12.00Fe8.00Nb8.00Zr48.00,0
4,Al10.00Ce60.00Cu20.00Ni10.00,3
5,Cu20.00Gd10.00Mg65.00Ni5.00,1
6,Ca55.00Cu20.00Mg25.00,1
7,Ag5.00Al12.50Cu15.00Fe5.00La62.50,3
8,B5.00C10.00Co35.00Fe40.00P10.00,2
9,Ca55.00Mg20.00Zn25.00,1


In [ ]:
# --- Flexible NN for D_max regression (5760 → 1) ---
class DmaxNet(nn.Module):
    def __init__(self, input_dim, hidden_layers, dropout_rate=0.3, activation="relu"):
        """
        Args:
            input_dim:      number of input features (5760)
            hidden_layers:  list of ints, e.g. [1024, 512, 256]
            dropout_rate:   dropout probability applied after each hidden layer
            activation:     "relu", "leaky_relu", "elu", or "selu"
        """
        super().__init__()
        act_fn = {"relu": nn.ReLU, "leaky_relu": nn.LeakyReLU,
                  "elu": nn.ELU, "selu": nn.SELU}[activation]

        layers = []
        prev = input_dim
        for units in hidden_layers:
            layers.append(nn.Linear(prev, units))
            layers.append(nn.BatchNorm1d(units))
            layers.append(act_fn())
            layers.append(nn.Dropout(dropout_rate))
            prev = units
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


def train_one_epoch(model, loader, optimizer, criterion, device):
    """Train model for one epoch. Returns average training loss."""
    model.train()
    total_loss = 0.0
    n_batches = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        n_batches += 1
    return total_loss / n_batches


def evaluate(model, loader, criterion, device):
    """Evaluate model on a loader. Returns average loss."""
    model.eval()
    total_loss = 0.0
    n_batches = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            loss = criterion(model(xb), yb)
            total_loss += loss.item()
            n_batches += 1
    return total_loss / n_batches


def predict(model, loader, device):
    """Run inference on a loader. Returns (predictions, actuals) if labels exist, else just (predictions, None)."""
    model.eval()
    all_preds = []
    all_actuals = []
    has_labels = False
    with torch.no_grad():
        for batch in loader:
            if isinstance(batch, (list, tuple)) and len(batch) >= 2:
                xb, yb = batch[0], batch[1]
                has_labels = True
                all_actuals.append(yb.cpu().numpy())
            else:
                xb = batch[0] if isinstance(batch, (list, tuple)) else batch
            xb = xb.to(device)
            all_preds.append(model(xb).cpu().numpy())
    preds = np.concatenate(all_preds)
    actuals = np.concatenate(all_actuals) if has_labels else None
    return preds, actuals


In [42]:
#define the evaluate parameters function for the ax optimization loop of the dmax_nn model
def evaluate_parameters_NN_dmax(parameters):
    
    #break down the parameters
    batch_size = parameters.get("batch_size", 64)
    n_layers = parameters.get("n_layers", 3)
    layer_1_dim = parameters.get("layer_1_dim", 1024)
    layer_2_dim = parameters.get("layer_2_dim", 1024)
    layer_3_dim = parameters.get("layer_3_dim", 512)
    dropout_rate = parameters.get("dropout_rate", 0.3)
    activation = parameters.get("activation", "relu")
    lr = parameters.get("lr", 1e-3)
    weight_decay = parameters.get("weight_decay", 1e-4)
    patience = parameters.get("early_stopping_patience", 20)
    
    #combine layer dimensions into a list for the model
    hidden_layers = [layer_1_dim, layer_2_dim, layer_3_dim][:n_layers]
    
    # Auto-detect device
    if torch.cuda.is_available():
        device = torch.device('cuda')
    elif torch.backends.mps.is_available():
        device = torch.device('mps')
    else:
        device = torch.device('cpu')
    print(f"Using device: {device}")
    
    
    #copy the x and y data
    y_data = y_train_opt.copy()
    x_data = X_train_opt.copy()
    
    #initialize cross fold info
    all_fold_mse = []
    all_fold_rmse = []
    all_fold_mae = []
    
    for fold in range(1,6):
        print(f"Starting fold {fold}...")
        
        #Establish from the folds the train and test data
        X_fold_train = x_data[cv_groups != (fold-1)]
        y_fold_train = np.array(y_data)[cv_groups != (fold-1)]
        
        X_fold_test = x_data[cv_groups == (fold-1)]
        y_fold_test = np.array(y_data)[cv_groups == (fold-1)]
        
        #split the fold train data into train and validation data
        X_fold_train, X_fold_val, y_fold_train, y_fold_val = train_test_split(X_fold_train, y_fold_train, test_size=0.2, random_state=42)
        
        #scale the X_data
        Scaler_X_fold = StandardScaler()
        X_fold_train_scaled = Scaler_X_fold.fit_transform(X_fold_train)
        X_fold_val_scaled = Scaler_X_fold.transform(X_fold_val)
        X_fold_test_scaled = Scaler_X_fold.transform(X_fold_test)
        
        #Scale the y data
        Scaler_y_fold = StandardScaler()
        y_fold_train_scaled = Scaler_y_fold.fit_transform(y_fold_train.reshape(-1, 1)).flatten()
        y_fold_val_scaled = Scaler_y_fold.transform(y_fold_val.reshape(-1, 1)).flatten()
        y_fold_test_scaled = Scaler_y_fold.transform(y_fold_test.reshape(-1, 1)).flatten()
        
        #convert all data to tensors
        X_fold_train_tensor = torch.tensor(X_fold_train_scaled, dtype=torch.float32).to(device)
        y_fold_train_tensor = torch.tensor(y_fold_train_scaled, dtype=torch.float32).to(device)
        X_fold_val_tensor = torch.tensor(X_fold_val_scaled, dtype=torch.float32).to(device)
        y_fold_val_tensor = torch.tensor(y_fold_val_scaled, dtype=torch.float32).to(device)
        X_fold_test_tensor = torch.tensor(X_fold_test_scaled, dtype=torch.float32).to(device)
        y_fold_test_tensor = torch.tensor(y_fold_test_scaled, dtype=torch.float32).to(device)
        
        #convert tensors to datasets then dataloaders
        train_ds = TensorDataset(X_fold_train_tensor, y_fold_train_tensor)
        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, generator=torch.Generator().manual_seed(42))
        val_ds = TensorDataset(X_fold_val_tensor, y_fold_val_tensor)
        val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=True, generator=torch.Generator().manual_seed(42)) 
        test_ds = TensorDataset(X_fold_test_tensor, y_fold_test_tensor)
        test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, generator=torch.Generator().manual_seed(42))    

        #initialize the model
        input_dim = X_fold_train_tensor.shape[1]
        model = DmaxNet(input_dim, hidden_layers, dropout_rate, activation).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=patience // 3, factor=0.5)
        criterion = nn.MSELoss()
        
        
        best_val_loss = float("inf")
        best_state = None
        wait = 0
        
        for epoch in range(1000):

            train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
            val_loss = evaluate(model, val_loader, criterion, device)
            if epoch % 10 == 0:
                print(f"  Epoch {epoch}... Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")
            scheduler.step(val_loss)
            
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                wait = 0
            else:
                wait += 1
                if wait >= patience:
                    break
        
        # Restore best model and evaluate on fold test set
        model.load_state_dict(best_state)
        
        #predict the fold test set and inverse transform the predictions and actuals back to original scale
        y_fold_test_pred_scaled, y_fold_test_actual_scaled = predict(model, test_loader, device)
        y_fold_test_pred = Scaler_y_fold.inverse_transform(y_fold_test_pred_scaled.reshape(-1, 1)).flatten()
        y_fold_test_actual = Scaler_y_fold.inverse_transform(y_fold_test_actual_scaled.reshape(-1, 1)).flatten()
        
        #calculate the fold mse, rmse, and mae and add to the list of fold metrics
        fold_mse = np.mean((y_fold_test_pred - y_fold_test_actual) ** 2)
        fold_rmse = np.sqrt(fold_mse)
        fold_mae = np.mean(np.abs(y_fold_test_pred - y_fold_test_actual))
        
        all_fold_mse.append(fold_mse)
        all_fold_rmse.append(fold_rmse)
        all_fold_mae.append(fold_mae)
        
        #print the fold metrics
        print(f"Fold {fold} - MSE: {fold_mse:.4f}, RMSE: {fold_rmse:.4f}, MAE: {fold_mae:.4f}")

    mean_rmse = np.mean(all_fold_rmse)
    mean_mse = np.mean(all_fold_mse)
    mean_mae = np.mean(all_fold_mae)
    print(f"\nMean CV MSE: {mean_mse:.4f}, Mean CV RMSE: {mean_rmse:.4f}, Mean CV MAE: {mean_mae:.4f}")
    return mean_rmse, sem(all_fold_rmse)


In [45]:
#initialize the ax client for the dmax_nn model
ax_client_dmax_nn = AxClient()
ax_client_dmax_nn.create_experiment(
    name="NN CBVF + CALPHAD to D_max Regression",
    parameters=[
        {
            "name": "dropout_rate",
            "type": "range",
            "bounds": [0.0, 0.5],
        },
        {
            "name": "activation",
            "type": "choice",
            "values": ["relu", "leaky_relu", "elu", "selu"],
            "is_ordered": False,
            "sort_values": False,
        },
        {
            "name": "weight_decay",
            "type": "range",
            "bounds": [1e-6, 1e-2],
            "log_scale": True,
        },
        {
            "name": "lr",
            "type": "range",
            "bounds": [1e-5, 1e-2],
            "log_scale": True,
        },
        {
            "name": "batch_size",
            "type": "choice",
            "values": [32, 64, 128, 256],
            "is_ordered": True,
            "sort_values": True,
        },
        {
            "name": "early_stopping_patience",
            "type": "range",
            "bounds": [10, 50],
            "value_type": "int",
        },
        {
            "name": "n_layers",
            "type": "choice",
            "values": [1, 2, 3],
            "value_type": "int",
            "is_ordered": True,
            "sort_values": True,
        },
        {
            "name": "hidden1",
            "type": "choice",
            "values": [128, 256, 512, 1024, 2048],
            "value_type": "int",
            "is_ordered": True,
            "sort_values": True,

        },
        {
            "name": "hidden2",
            "type": "choice",
            "values": [64, 128, 256, 512, 1024],
            "value_type": "int",
            "is_ordered": True,
            "sort_values": True,

        },
        {
            "name": "hidden3",
            "type": "choice",
            "values": [32, 64, 128, 256, 512],
            "value_type": "int",
            "is_ordered": True,
            "sort_values": True,
        }
        
    ],
    objectives={"avg_rmse_nonzero": ObjectiveProperties(minimize=True)}
)

[INFO 04-02 09:54:19] ax.service.utils.instantiation: Inferred value type of ParameterType.FLOAT for parameter dropout_rate. If that is not the expected value type, you can explicitly specify 'value_type' ('int', 'float', 'bool' or 'str') in parameter dict.
[INFO 04-02 09:54:19] ax.service.utils.instantiation: Inferred value type of ParameterType.STRING for parameter activation. If that is not the expected value type, you can explicitly specify 'value_type' ('int', 'float', 'bool' or 'str') in parameter dict.
[INFO 04-02 09:54:19] ax.service.utils.instantiation: Inferred value type of ParameterType.FLOAT for parameter weight_decay. If that is not the expected value type, you can explicitly specify 'value_type' ('int', 'float', 'bool' or 'str') in parameter dict.
[INFO 04-02 09:54:19] ax.service.utils.instantiation: Inferred value type of ParameterType.FLOAT for parameter lr. If that is not the expected value type, you can explicitly specify 'value_type' ('int', 'float', 'bool' or 'str'

In [46]:
#Perform the ax optimization loop for the dmax_nn model

#initialize the save path for the dmax_nn ax client
ax_client_dmax_nn_save_path = r"Ax_checkpoints\ax_client_dmax_nn_checkpoint.json"

#check if there is an existing checkpoint and load it, otherwise start a new optimization loop
try:
    ax_client_dmax_nn = AxClient.load_from_json_file(filepath=ax_client_dmax_nn_save_path)
    print(f"Loaded existing Ax client checkpoint from {ax_client_dmax_nn_save_path}")
    
    completed_trials = len(ax_client_dmax_nn.experiment.trials)
    target_trials = 100
    remaining_trials = target_trials - completed_trials
    
    print(f"Loaded {completed_trials} completed trials from ax_client_dmax_nn.json")
    print(f"Remaining trials to reach {target_trials}: {remaining_trials}")
    
    for i in range(remaining_trials):
        parameters, trial_index = ax_client_dmax_nn.get_next_trial()
        print(f"\nStarting trial {trial_index} with parameters: {parameters}")
        mean_rmse, rmse_sem = evaluate_parameters_NN_dmax(parameters)
        ax_client_dmax_nn.complete_trial(trial_index=trial_index, raw_data={"avg_rmse_nonzero": (mean_rmse, rmse_sem)})
        print(f"Completed trial {trial_index} with mean RMSE: {mean_rmse:.4f} ± {rmse_sem:.4f}")
        
        # Save the Ax client state after each trial
        ax_client_dmax_nn.save_to_json_file(filepath=ax_client_dmax_nn_save_path)
        print(f"Saved Ax client checkpoint")
    
    
except Exception as e:
    print(f"No existing checkpoint found, starting new optimization loop. Error: {e}")
    
    for i in range(100):
        parameters, trial_index = ax_client_dmax_nn.get_next_trial()
        print(f"\nStarting trial {trial_index} with parameters: {parameters}")
        mean_rmse, rmse_sem = evaluate_parameters_NN_dmax(parameters)
        ax_client_dmax_nn.complete_trial(trial_index=trial_index, raw_data={"avg_rmse_nonzero": (mean_rmse, rmse_sem)})
        print(f"Completed trial {trial_index} with mean RMSE: {mean_rmse:.4f} ± {rmse_sem:.4f}")
        
        # Save the Ax client state after each trial
        ax_client_dmax_nn.save_to_json_file(filepath=ax_client_dmax_nn_save_path)
        print(f"Saved Ax client checkpoint")

[INFO 04-02 09:54:21] ax.service.ax_client: Generated new trial 0 with parameters {'dropout_rate': 0.323671, 'weight_decay': 0.003794, 'lr': 0.003346, 'batch_size': 64, 'early_stopping_patience': 29, 'n_layers': 1, 'hidden1': 256, 'hidden2': 128, 'hidden3': 256, 'activation': 'elu'} using model Sobol.


No existing checkpoint found, starting new optimization loop. Error: [Errno 2] No such file or directory: 'Ax_checkpoints\\ax_client_dmax_nn_checkpoint.json'

Starting trial 0 with parameters: {'dropout_rate': 0.32367146015167236, 'weight_decay': 0.003793809981931019, 'lr': 0.003345730936533549, 'batch_size': 64, 'early_stopping_patience': 29, 'n_layers': 1, 'hidden1': 256, 'hidden2': 128, 'hidden3': 256, 'activation': 'elu'}
Using device: cuda
Starting fold 1...
  Epoch 0... Train Loss: 20.6938, Val Loss: 7.6735
  Epoch 10... Train Loss: 0.4930, Val Loss: 0.4625
  Epoch 20... Train Loss: 0.5059, Val Loss: 0.5263
  Epoch 30... Train Loss: 0.4543, Val Loss: 0.3365
  Epoch 40... Train Loss: 0.3735, Val Loss: 0.3752
Fold 1 - MSE: 41.5238, RMSE: 6.4439, MAE: 4.3056
Starting fold 2...
  Epoch 0... Train Loss: 16.1146, Val Loss: 6.6206
  Epoch 10... Train Loss: 0.5687, Val Loss: 0.7827
  Epoch 20... Train Loss: 0.4337, Val Loss: 0.6551
  Epoch 30... Train Loss: 0.3941, Val Loss: 0.6522
  Epo

[INFO 04-02 09:54:33] ax.service.ax_client: Completed trial 0 with data: {'avg_rmse_nonzero': 7.378909}.
[INFO 04-02 09:54:33] ax.service.ax_client: Generated new trial 1 with parameters {'dropout_rate': 0.0562, 'weight_decay': 5e-05, 'lr': 1.9e-05, 'batch_size': 256, 'early_stopping_patience': 38, 'n_layers': 3, 'hidden1': 512, 'hidden2': 512, 'hidden3': 64, 'activation': 'relu'} using model Sobol.


Fold 5 - MSE: 23.3462, RMSE: 4.8318, MAE: 3.8941

Mean CV MSE: 60.6212, Mean CV RMSE: 7.3789, Mean CV MAE: 5.1547
Completed trial 0 with mean RMSE: 7.3789 ± 1.2423
Saved Ax client checkpoint

Starting trial 1 with parameters: {'dropout_rate': 0.05619959533214569, 'weight_decay': 4.9869540134627203e-05, 'lr': 1.9112885152396023e-05, 'batch_size': 256, 'early_stopping_patience': 38, 'n_layers': 3, 'hidden1': 512, 'hidden2': 512, 'hidden3': 64, 'activation': 'relu'}
Using device: cuda
Starting fold 1...
  Epoch 0... Train Loss: 1.2494, Val Loss: 0.9071
  Epoch 10... Train Loss: 0.4265, Val Loss: 0.5038
  Epoch 20... Train Loss: 0.3628, Val Loss: 0.2639
  Epoch 30... Train Loss: 0.2964, Val Loss: 0.3173
  Epoch 40... Train Loss: 0.2234, Val Loss: 0.2799
  Epoch 50... Train Loss: 0.3026, Val Loss: 0.2923
Fold 1 - MSE: 36.3759, RMSE: 6.0312, MAE: 4.2530
Starting fold 2...
  Epoch 0... Train Loss: 0.9090, Val Loss: 1.0626
  Epoch 10... Train Loss: 0.3368, Val Loss: 0.5440
  Epoch 20... Train 

[INFO 04-02 09:54:43] ax.service.ax_client: Completed trial 1 with data: {'avg_rmse_nonzero': 7.37515}.
[INFO 04-02 09:54:43] ax.service.ax_client: Generated new trial 2 with parameters {'dropout_rate': 0.140552, 'weight_decay': 4e-06, 'lr': 0.000468, 'batch_size': 128, 'early_stopping_patience': 40, 'n_layers': 2, 'hidden1': 2048, 'hidden2': 512, 'hidden3': 512, 'activation': 'leaky_relu'} using model Sobol.


...

  Epoch 200... Train Loss: 0.1524, Val Loss: 0.6960
Fold 5 - MSE: 18.2659, RMSE: 4.2739, MAE: 3.6958

Mean CV MSE: 30.9247, Mean CV RMSE: 5.4932, Mean CV MAE: 3.8615
Completed trial 98 with mean RMSE: 5.4932 ± 0.4329
Saved Ax client checkpoint


[INFO 04-02 10:21:56] ax.service.ax_client: Generated new trial 99 with parameters {'dropout_rate': 0.0, 'weight_decay': 0.005411, 'lr': 0.002379, 'batch_size': 256, 'early_stopping_patience': 12, 'n_layers': 3, 'hidden1': 2048, 'hidden2': 1024, 'hidden3': 128, 'activation': 'leaky_relu'} using model BoTorch.



Starting trial 99 with parameters: {'dropout_rate': 0.0, 'weight_decay': 0.005411252119188789, 'lr': 0.0023786499051302295, 'batch_size': 256, 'early_stopping_patience': 12, 'n_layers': 3, 'hidden1': 2048, 'hidden2': 1024, 'hidden3': 128, 'activation': 'leaky_relu'}
Using device: cuda
Starting fold 1...
  Epoch 0... Train Loss: 2.3404, Val Loss: 29.6148
  Epoch 10... Train Loss: 0.6567, Val Loss: 0.6541
  Epoch 20... Train Loss: 0.4761, Val Loss: 0.3791
  Epoch 30... Train Loss: 0.3592, Val Loss: 0.4276
Fold 1 - MSE: 36.6281, RMSE: 6.0521, MAE: 4.0636
Starting fold 2...
  Epoch 0... Train Loss: 6.5628, Val Loss: 9.1879
  Epoch 10... Train Loss: 0.5504, Val Loss: 0.7018
  Epoch 20... Train Loss: 0.2783, Val Loss: 0.4806
  Epoch 30... Train Loss: 0.2761, Val Loss: 0.3779
  Epoch 40... Train Loss: 0.1998, Val Loss: 0.3660
Fold 2 - MSE: 37.8337, RMSE: 6.1509, MAE: 3.8428
Starting fold 3...
  Epoch 0... Train Loss: 6.8919, Val Loss: 23.0733
  Epoch 10... Train Loss: 0.4890, Val Loss: 0.714

[INFO 04-02 10:22:06] ax.service.ax_client: Completed trial 99 with data: {'avg_rmse_nonzero': 5.642336}.


  Epoch 50... Train Loss: 0.1749, Val Loss: 0.7284
Fold 5 - MSE: 7.0162, RMSE: 2.6488, MAE: 2.0585

Mean CV MSE: 34.6783, Mean CV RMSE: 5.6423, Mean CV MAE: 3.7217
Completed trial 99 with mean RMSE: 5.6423 ± 0.8430
Saved Ax client checkpoint


In [48]:
# Get best parameters after all trials
best_parameters, values = ax_client_dmax_nn.get_best_parameters()
print(f"\n{'='*60}")
print("OPTIMIZATION COMPLETE")
print('='*60)
print(f"Best parameters: {best_parameters}")
print(f"Best Non-zero RMSE: {values[0]['avg_rmse_nonzero']:.4f}")


OPTIMIZATION COMPLETE
Best parameters: {'dropout_rate': 0.07095621515728066, 'weight_decay': 1e-06, 'lr': 0.0030231286188883797, 'batch_size': 256, 'early_stopping_patience': 18, 'n_layers': 3, 'hidden1': 2048, 'hidden2': 1024, 'hidden3': 512, 'activation': 'leaky_relu'}
Best Non-zero RMSE: 5.3189


In [49]:
#evaluate the best parameters on the test set

# Auto-detect device
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f"Using device: {device}")

#best parameters from the ax optimization loop
best_parameters_NN_dmax = {'dropout_rate': 0.07095621515728066, 'weight_decay': 1e-06, 'lr': 0.0030231286188883797, 'batch_size': 256, 'early_stopping_patience': 18, 'n_layers': 3, 'hidden1': 2048, 'hidden2': 1024, 'hidden3': 512, 'activation': 'leaky_relu'}

#combine layer dimensions into a list for the model
hidden_layers = [best_parameters_NN_dmax["hidden1"], best_parameters_NN_dmax["hidden2"], best_parameters_NN_dmax["hidden3"]][:best_parameters_NN_dmax["n_layers"]]

#copy the x and y data
X_evaluate_train = X_train_opt.copy()
y_evaluate_train = y_train_opt.copy()
X_evaluate_test = X_test.copy() 
y_evaluate_test = y_test.copy()

#split the train data into train and validation data
X_evaluate_train, X_evaluate_val, y_evaluate_train, y_evaluate_val = train_test_split(X_evaluate_train, y_evaluate_train, test_size=0.2, random_state=42)

#scale the X_data
Scaler_X_evaluate = StandardScaler()
X_evaluate_train_scaled = Scaler_X_evaluate.fit_transform(X_evaluate_train)
X_evaluate_val_scaled = Scaler_X_evaluate.transform(X_evaluate_val)
X_evaluate_test_scaled = Scaler_X_evaluate.transform(X_evaluate_test)

#Scale the y data
Scaler_y_evaluate = StandardScaler()
y_evaluate_train_scaled = Scaler_y_evaluate.fit_transform(np.array(y_evaluate_train).reshape(-1, 1)).flatten()
y_evaluate_val_scaled = Scaler_y_evaluate.transform(np.array(y_evaluate_val).reshape(-1, 1)).flatten()
y_evaluate_test_scaled = Scaler_y_evaluate.transform(np.array(y_evaluate_test).reshape(-1, 1)).flatten()

#convert all data to tensors
X_evaluate_train_tensor = torch.tensor(X_evaluate_train_scaled, dtype=torch.float32).to(device)
y_evaluate_train_tensor = torch.tensor(y_evaluate_train_scaled, dtype=torch.float32).to(device)
X_evaluate_val_tensor = torch.tensor(X_evaluate_val_scaled, dtype=torch.float32).to(device)
y_evaluate_val_tensor = torch.tensor(y_evaluate_val_scaled, dtype=torch.float32).to(device)
X_evaluate_test_tensor = torch.tensor(X_evaluate_test_scaled, dtype=torch.float32).to(device)
y_evaluate_test_tensor = torch.tensor(y_evaluate_test_scaled, dtype=torch.float32).to(device)

#convert tensors to datasets then dataloaders
evaluate_train_ds = TensorDataset(X_evaluate_train_tensor, y_evaluate_train_tensor)
evaluate_train_loader = DataLoader(evaluate_train_ds, batch_size=best_parameters_NN_dmax["batch_size"], shuffle=True, generator=torch.Generator().manual_seed(42))
evaluate_val_ds = TensorDataset(X_evaluate_val_tensor, y_evaluate_val_tensor)
evaluate_val_loader = DataLoader(evaluate_val_ds, batch_size=best_parameters_NN_dmax["batch_size"], shuffle=True, generator=torch.Generator().manual_seed(42))
evaluate_test_ds = TensorDataset(X_evaluate_test_tensor, y_evaluate_test_tensor)
evaluate_test_loader = DataLoader(evaluate_test_ds, batch_size=best_parameters_NN_dmax["batch_size"], shuffle=False, generator=torch.Generator().manual_seed(42))


#initialize the model
input_dim = X_evaluate_train_tensor.shape[1]
model = DmaxNet(input_dim, hidden_layers, best_parameters_NN_dmax["dropout_rate"], best_parameters_NN_dmax["activation"]).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=best_parameters_NN_dmax["lr"], weight_decay=best_parameters_NN_dmax["weight_decay"])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=best_parameters_NN_dmax["early_stopping_patience"] // 3, factor=0.5)
criterion = nn.MSELoss()


best_val_loss = float("inf")
best_state = None
wait = 0

for epoch in range(1000):

    train_loss = train_one_epoch(model, evaluate_train_loader, optimizer, criterion, device)
    val_loss = evaluate(model, evaluate_val_loader, criterion, device)
    if epoch % 10 == 0:
        print(f"  Epoch {epoch}... Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")
    scheduler.step(val_loss)
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        wait = 0
    else:
        wait += 1
        if wait >= best_parameters_NN_dmax["early_stopping_patience"]:
            break

# Restore best model and evaluate on fold test set
model.load_state_dict(best_state)

#predict the fold test set and inverse transform the predictions and actuals back to original scale
y_fold_test_pred_scaled, y_fold_test_actual_scaled = predict(model, evaluate_test_loader, device)
y_fold_test_pred = Scaler_y_evaluate.inverse_transform(y_fold_test_pred_scaled.reshape(-1, 1)).flatten()
y_fold_test_actual = Scaler_y_evaluate.inverse_transform(y_fold_test_actual_scaled.reshape(-1, 1)).flatten()

#calculate the fold mse, rmse, and mae and add to the list of fold metrics
fold_mse = np.mean((y_fold_test_pred - y_fold_test_actual) ** 2)
fold_rmse = np.sqrt(fold_mse)
fold_mae = np.mean(np.abs(y_fold_test_pred - y_fold_test_actual))

#print the fold metrics
print(f"Test Results - MSE: {fold_mse:.4f}, RMSE: {fold_rmse:.4f}, MAE: {fold_mae:.4f}")


Using device: cuda
  Epoch 0... Train Loss: 8.4527, Val Loss: 287.3803
  Epoch 10... Train Loss: 0.7221, Val Loss: 0.5460
  Epoch 20... Train Loss: 0.4962, Val Loss: 0.3565
  Epoch 30... Train Loss: 0.2973, Val Loss: 0.3374
  Epoch 40... Train Loss: 0.2423, Val Loss: 0.3463
  Epoch 50... Train Loss: 0.2020, Val Loss: 0.3327
  Epoch 60... Train Loss: 0.1806, Val Loss: 0.3323
Test Results - MSE: 10.8110, RMSE: 3.2880, MAE: 2.2005


In [57]:
#Define the XGBboost parameter evaluation function for the ax optimization loop of the dmax_xgb model
def evaluate_parameters_XGB_dmax(parameters):
    
    #pull the relevent parameters for the xgboost model from the input parameters
    n_estimators = parameters.get("n_estimators", 100)
    max_depth = parameters.get("max_depth", 6)
    learning_rate = parameters.get("learning_rate", 0.1)
    subsample = parameters.get("subsample", 1.0)
    colsample_bytree = parameters.get("colsample_bytree", 1.0)
    minchild_weight = parameters.get("min_child_weight", 1)
    reg_alpha = parameters.get("reg_alpha", 0)
    reg_lambda = parameters.get("reg_lambda", 1)
    gamma = parameters.get("gamma", 0)
    earlystopping_rounds = parameters.get("early_stopping_rounds", 10)

    
    # Auto-detect device
    if torch.cuda.is_available():
        device = torch.device('cuda')
    elif torch.backends.mps.is_available():
        device = torch.device('mps')
    else:
        device = torch.device('cpu')
    print(f"Using device: {device}")
    
    
    #copy the x and y data
    y_data = y_train_opt.copy()
    x_data = X_train_opt.copy()
    
    #initialize cross fold info
    all_fold_mse = []
    all_fold_rmse = []
    all_fold_mae = []
    
    for fold in range(1,6):
        print(f"Starting fold {fold}...")
        
        #Establish from the folds the train and test data
        X_fold_train = x_data[cv_groups != (fold-1)]
        y_fold_train = np.array(y_data)[cv_groups != (fold-1)]
        
        X_fold_test = x_data[cv_groups == (fold-1)]
        y_fold_test = np.array(y_data)[cv_groups == (fold-1)]
        
        #split the fold train data into train and validation data
        X_fold_train, X_fold_val, y_fold_train, y_fold_val = train_test_split(X_fold_train, y_fold_train, test_size=0.2, random_state=42)
        
        #itialize the xgboost model with the input parameters
        xgb_model = XGBRegressor(
            n_estimators=n_estimators,
            max_depth=max_depth,
            learning_rate=learning_rate,
            subsample=subsample,
            colsample_bytree=colsample_bytree,
            min_child_weight=minchild_weight,
            reg_alpha=reg_alpha,
            reg_lambda=reg_lambda,
            gamma=gamma,
            early_stopping_rounds=earlystopping_rounds,
            random_state=42,
            tree_method="hist",
            device = "cuda"
        )  
        
        #fit the xgboost model with early stopping
        xgb_model.fit(
            X_fold_train, y_fold_train,
            eval_set=[(X_fold_val, y_fold_val)],
            verbose=False
        )
        
        #predict the fold test set
        y_fold_test_pred = xgb_model.predict(X_fold_test)
        y_fold_test_actual = y_fold_test
        
        #calculate the fold mse, rmse, and mae and add to the list of fold metrics
        fold_mse = np.mean((y_fold_test_pred - y_fold_test_actual) ** 2)
        fold_rmse = np.sqrt(fold_mse)
        fold_mae = np.mean(np.abs(y_fold_test_pred - y_fold_test_actual))
        
        all_fold_mse.append(fold_mse)
        all_fold_rmse.append(fold_rmse)
        all_fold_mae.append(fold_mae)
        
        #print the fold metrics
        print(f"Fold {fold} - MSE: {fold_mse:.4f}, RMSE: {fold_rmse:.4f}, MAE: {fold_mae:.4f}")

    mean_rmse = np.mean(all_fold_rmse)
    mean_mse = np.mean(all_fold_mse)
    mean_mae = np.mean(all_fold_mae)
    print(f"\nMean CV MSE: {mean_mse:.4f}, Mean CV RMSE: {mean_rmse:.4f}, Mean CV MAE: {mean_mae:.4f}")
    return mean_rmse, sem(all_fold_rmse)




In [58]:
#define the ax client for the xgboost model
ax_client_xgb_dmax = AxClient()
ax_client_xgb_dmax.create_experiment(
    name="XGB opt CBFV CALPHAD to D_max Regression",
    parameters=[
        {
            "name": "n_estimators",
            "type": "range",
            "bounds": [100, 10000],
            "value_type": "int",
        },
        {
            "name": "max_depth",
            "type": "range",
            "bounds": [3, 100],
            "value_type": "int",
        },
        {
            "name": "learning_rate",
            "type": "range",
            "bounds": [0.005, 0.3],
            "value_type": "float",
            "log_scale": True,
        },
        {
            "name": "subsample",
            "type": "range",
            "bounds": [0.5, 1.0],
            "value_type": "float",

        },
        {
            "name": "colsample_bytree",
            "type": "range",
            "bounds": [0.3, 1.0],
            "value_type": "float",
        },
        {
            "name": "min_child_weight",
            "type": "range",
            "bounds": [1, 20],
            "value_type": "int",
        },
        {
            "name": "reg_alpha",
            "type": "range",
            "bounds": [1e-6, 10.0],
            "value_type": "float",
            "log_scale": True,
        },
        {
            "name": "reg_lambda",
            "type": "range",
            "bounds": [1e-6, 10.0],
            "value_type": "float",
            "log_scale": True,
        },
        {
            "name": "gamma",
            "type": "range",
            "bounds": [1e-6, 5.0],
            "log_scale": True,
            "value_type": "float",
        },
        {
            "name": "early_stopping_rounds",
            "type": "range",
            "bounds": [10, 50],
            "value_type": "int",
        },
    ],
    objectives={"avg_rmse_nonzero": ObjectiveProperties(minimize=True)}
)

[INFO 04-02 10:43:43] ax.generation_strategy.dispatch_utils: Using Generators.BOTORCH_MODULAR since there is at least one ordered parameter and there are no unordered categorical parameters.
[INFO 04-02 10:43:43] ax.generation_strategy.dispatch_utils: Using Bayesian Optimization generation strategy: GenerationStrategy(name='Sobol+BoTorch', steps=[Sobol for 20 trials, BoTorch for subsequent trials]). Iterations after 20 will take longer to generate due to model-fitting.


In [59]:
#Perform the ax optimization loop for the xgboost model
#initialize the save path for the xgboost ax client
ax_client_xgb_dmax_save_path = r"Ax_checkpoints\ax_client_xgb_dmax_checkpoint.json"

#check if there is an existing checkpoint and load it, otherwise start a new optimization loop
try:
    ax_client_xgb_dmax = AxClient.load_from_json_file(filepath=ax_client_xgb_dmax_save_path)
    print(f"Loaded existing Ax client checkpoint from {ax_client_xgb_dmax_save_path}")
    
    completed_trials = len(ax_client_xgb_dmax.experiment.trials)
    target_trials = 100
    remaining_trials = target_trials - completed_trials
    
    print(f"Loaded {completed_trials} completed trials from ax_client_xgb_dmax.json")
    print(f"Remaining trials to reach {target_trials}: {remaining_trials}")
    
    for i in range(remaining_trials):
        parameters, trial_index = ax_client_xgb_dmax.get_next_trial()
        print(f"\nStarting trial {trial_index} with parameters: {parameters}")
        mean_rmse, rmse_sem = evaluate_parameters_XGB_dmax(parameters)
        ax_client_xgb_dmax.complete_trial(trial_index=trial_index, raw_data={"avg_rmse_nonzero": (mean_rmse, rmse_sem)})
        print(f"Completed trial {trial_index} with mean RMSE: {mean_rmse:.4f} ± {rmse_sem:.4f}")
        
        # Save the Ax client state after each trial
        ax_client_xgb_dmax.save_to_json_file(filepath=ax_client_xgb_dmax_save_path)
        print(f"Saved Ax client checkpoint")
    
    
except Exception as e:
    print(f"No existing checkpoint found, starting new optimization loop. Error: {e}")
    
    for i in range(100):
        parameters, trial_index = ax_client_xgb_dmax.get_next_trial()
        print(f"\nStarting trial {trial_index} with parameters: {parameters}")
        mean_rmse, rmse_sem = evaluate_parameters_XGB_dmax(parameters)
        ax_client_xgb_dmax.complete_trial(trial_index=trial_index, raw_data={"avg_rmse_nonzero": (mean_rmse, rmse_sem)})
        print(f"Completed trial {trial_index} with mean RMSE: {mean_rmse:.4f} ± {rmse_sem:.4f}")
        
        # Save the Ax client state after each trial
        ax_client_xgb_dmax.save_to_json_file(filepath=ax_client_xgb_dmax_save_path)
        print(f"Saved Ax client checkpoint")

[INFO 04-02 10:43:44] ax.service.ax_client: Generated new trial 0 with parameters {'n_estimators': 5263, 'max_depth': 64, 'learning_rate': 0.006597, 'subsample': 0.522859, 'colsample_bytree': 0.462448, 'min_child_weight': 4, 'reg_alpha': 2e-05, 'reg_lambda': 0.029245, 'gamma': 2.7e-05, 'early_stopping_rounds': 43} using model Sobol.


No existing checkpoint found, starting new optimization loop. Error: [Errno 2] No such file or directory: 'Ax_checkpoints\\ax_client_xgb_dmax_checkpoint.json'

Starting trial 0 with parameters: {'n_estimators': 5263, 'max_depth': 64, 'learning_rate': 0.006596605959109167, 'subsample': 0.5228592325001955, 'colsample_bytree': 0.46244767457246777, 'min_child_weight': 4, 'reg_alpha': 1.95441553706182e-05, 'reg_lambda': 0.029244928589894813, 'gamma': 2.686119407922972e-05, 'early_stopping_rounds': 43}
Using device: cuda
Starting fold 1...


c:\Users\Chris\Documents\ML_Project\ML_GFA_env\Lib\site-packages\xgboost\core.py:774: UserWarning:

[10:44:28] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.




Fold 1 - MSE: 36.8088, RMSE: 6.0670, MAE: 3.9734
Starting fold 2...
Fold 2 - MSE: 36.0286, RMSE: 6.0024, MAE: 3.8601
Starting fold 3...
Fold 3 - MSE: 8.0210, RMSE: 2.8321, MAE: 2.0930
Starting fold 4...
Fold 4 - MSE: 28.0342, RMSE: 5.2947, MAE: 4.4408
Starting fold 5...


[INFO 04-02 10:48:35] ax.service.ax_client: Completed trial 0 with data: {'avg_rmse_nonzero': 4.431959}.
[INFO 04-02 10:48:36] ax.service.ax_client: Generated new trial 1 with parameters {'n_estimators': 3102, 'max_depth': 7, 'learning_rate': 0.281275, 'subsample': 0.986166, 'colsample_bytree': 0.788279, 'min_child_weight': 12, 'reg_alpha': 0.265844, 'reg_lambda': 9e-06, 'gamma': 0.371676, 'early_stopping_rounds': 15} using model Sobol.


...

Fold 5 - MSE: 3.2620, RMSE: 1.8061, MAE: 1.3854

Mean CV MSE: 22.9325, Mean CV RMSE: 4.4481, Mean CV MAE: 3.1371
Completed trial 98 with mean RMSE: 4.4481 ± 0.8870
Saved Ax client checkpoint


[INFO 04-02 15:47:07] ax.service.ax_client: Generated new trial 99 with parameters {'n_estimators': 7573, 'max_depth': 3, 'learning_rate': 0.005, 'subsample': 0.5, 'colsample_bytree': 0.978954, 'min_child_weight': 12, 'reg_alpha': 5.8e-05, 'reg_lambda': 1e-06, 'gamma': 0.000862, 'early_stopping_rounds': 50} using model BoTorch.



Starting trial 99 with parameters: {'n_estimators': 7573, 'max_depth': 3, 'learning_rate': 0.005, 'subsample': 0.5, 'colsample_bytree': 0.9789541194089539, 'min_child_weight': 12, 'reg_alpha': 5.801642038892301e-05, 'reg_lambda': 1e-06, 'gamma': 0.0008623561773663504, 'early_stopping_rounds': 50}
Using device: cuda
Starting fold 1...
Fold 1 - MSE: 36.8340, RMSE: 6.0691, MAE: 3.8568
Starting fold 2...
Fold 2 - MSE: 39.3644, RMSE: 6.2741, MAE: 3.9132
Starting fold 3...
Fold 3 - MSE: 8.0673, RMSE: 2.8403, MAE: 1.9161
Starting fold 4...
Fold 4 - MSE: 30.1692, RMSE: 5.4926, MAE: 4.5883
Starting fold 5...


[INFO 04-02 15:52:02] ax.service.ax_client: Completed trial 99 with data: {'avg_rmse_nonzero': 4.538462}.


Fold 5 - MSE: 4.0648, RMSE: 2.0161, MAE: 1.5896

Mean CV MSE: 23.7000, Mean CV RMSE: 4.5385, Mean CV MAE: 3.1728
Completed trial 99 with mean RMSE: 4.5385 ± 0.8807
Saved Ax client checkpoint


In [60]:
# Get best parameters after all trials
best_parameters, values = ax_client_xgb_dmax.get_best_parameters()
print(f"\n{'='*60}")
print("OPTIMIZATION COMPLETE")
print('='*60)
print(f"Best parameters: {best_parameters}")
print(f"Best Non-zero RMSE: {values[0]['avg_rmse_nonzero']:.4f}")


OPTIMIZATION COMPLETE
Best parameters: {'n_estimators': 9977, 'max_depth': 16, 'learning_rate': 0.015631890680163428, 'subsample': 0.5116966020773396, 'colsample_bytree': 0.3, 'min_child_weight': 20, 'reg_alpha': 2.32790009104675e-05, 'reg_lambda': 0.004199372127688964, 'gamma': 5.0, 'early_stopping_rounds': 10}
Best Non-zero RMSE: 4.3972
